# NIT-4 · 1 — The question, before there is an experiment to answer it

**The situation.** It is 1968. Congress is looking at a **negative income tax**: a
family that earns nothing receives a guarantee `G`, stated as a fraction of the
poverty line, and every dollar it earns takes `t` cents of that guarantee away. The
plan on the table is `G = 0.75`, `t = 0.50` — a transfer of 3,000 USD at zero
earnings, gone by the time a family earns 6,000. Five and a half million families
would be eligible.

The Treasury's objection is not the transfer. It is what the transfer does to work:
every hour a family head stops working is an hour of wages nobody taxes and an hour
the guarantee has to replace. The memorandum on the Secretary's desk says the plan
is affordable if it costs **fewer than 150 hours a year per family**, and nobody knows
whether it does. The literature says anywhere from nothing to a third of the working
year, and every number in it was read off families who were never assigned anything.

This notebook does the part that comes before any experiment. It writes the decision
down as an object, asks what the survey record can and cannot say, declares the
quantity an experiment would have to estimate, and prices the uncertainty — so that
notebook 2 can design the experiments that uncertainty justifies, and notebooks 3
and 4 can run four of them and pool what they found.

| | reaches into |
|---|---|
| the plan as arithmetic, and the decision | `design.DecisionSpec`, `design.ValuePerOutcome` |
| what the survey record supports | `identify.ols`, `identify.CausalGraph`, `identify.identify` |
| the quantity to be estimated | `estimands.Estimand`, `core.Intervention`, `core.Population`, `core.TimeWindow` |
| what resolving it is worth | `design.prior_from_history`, `design.evpi_gaussian`, `design.evoi_gaussian` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd

import guarantee as W
from axiom.core import D, Intervention
from axiom.design import (
    DecisionSpec, EVOIResult, StudySummary, ValuePerOutcome, evoi_gaussian, evpi_gaussian,
    prior_from_history,
)
from axiom.identify import CausalGraph, LinearEstimate, adjustment_sets, identify, ols, requires_unmeasured
from axiom.viz import causal_graph
from axiom.display import enable, table

from _style import annotate, caption, compare, density, figure, lines, mark_x, mark_y, shade

enable();  # every axiom result renders itself from here on

G_NAT, T_NAT = W.NATIONAL_PLAN
print(f"poverty line          {W.POVERTY_LINE:>12,.0f} USD (1970, family of four)")
print(f"eligible families     {W.N_ELIGIBLE:>12,d}")
print(f"the plan on the table G = {G_NAT}, t = {T_NAT}: "
      f"{G_NAT * W.POVERTY_LINE:,.0f} USD at zero earnings, break-even at "
      f"{W.break_even(G_NAT, T_NAT):,.0f} USD")
print(f"affordable if a family works fewer than {W.HOURS_THRESHOLD:.0f} hours a year less")

## 1 · The plan, as arithmetic

A plan is two numbers and a straight line. The transfer is `max(G·P − t·y, 0)`, so
the guarantee sets the height at zero earnings and the tax rate sets the slope. Both
are dimensionless — a fraction of a poverty line, a fraction of a dollar — and the whole
case study keeps them that way: the surface in notebook 5 is written over `(G, t)`,
and the operational readings (the transfer, the break-even) are one multiplication
away on every axis.

Nothing about the transfer schedule needs an experiment. What it costs depends on
where eligible families' earnings sit, and the programme owns that arithmetic.

In [ ]:
earnings = np.linspace(0.0, 9_000.0, 200)
schedules = {
    f"G = 0.50, t = 0.50": W.transfer(0.5, 0.5, earnings),
    f"G = {G_NAT:.2f}, t = {T_NAT:.2f}  (the plan on the table)": W.transfer(G_NAT, T_NAT, earnings),
    f"G = 0.75, t = 0.70": W.transfer(0.75, 0.7, earnings),
    f"G = 1.00, t = 0.50": W.transfer(1.0, 0.5, earnings),
}
fig = lines(
    earnings, schedules,
    title="A plan is a height and a slope",
    subtitle="the transfer a family receives against what it earns, for four plans on the lattice",
    x_title="family earnings (USD/year)", y_title="transfer (USD/year)",
)
mark_x(fig, W.POVERTY_LINE, text="poverty line")
caption(fig, "The guarantee sets where a line starts and the tax rate sets how fast it falls. "
             "Two plans with the same guarantee and different tax rates pay the same family "
             "at zero earnings and very different families at 4,000 USD — which is why the "
             "dose is two numbers and not one generosity index.")

rows = []
for g, t in ((0.0, 0.0), (0.5, 0.5), (G_NAT, T_NAT), (0.75, 0.7), (1.0, 0.5), (1.4, 0.8)):
    cost = W.plan_cost(g, t)
    rows.append([
        "no plan (control)" if g == 0.0 else W.plan_label(g, t),
        f"{g * W.POVERTY_LINE:,.0f}" if g else "—",
        f"{W.break_even(g, t):,.0f}" if g else "—",
        f"{cost:,.0f}",
        f"{cost * W.N_ELIGIBLE / 1e9:,.1f} bn" if g else "—",
    ])
table(rows, headers=("plan", "transfer at zero earnings (USD)", "break-even (USD)",
                     "cost per family-year (USD)", "national cost per year"))
national_cost = W.plan_cost(G_NAT, T_NAT) * W.N_ELIGIBLE
print(f"\nthe plan on the table costs {national_cost / 1e9:,.1f} bn USD a year before anybody changes how much they work")

## 2 · The decision, as an object

The Treasury's rule is a threshold on one number: the change in annual hours worked
by a family head under the plan, against no plan. The value of each hour is what the
memorandum says it is — the transfer that replaces it plus the tax it no longer pays —
and it is attached to the decision with its provenance, because notebook 5 is going to
ask what happens when that number is wrong.

`DecisionSpec` pays `value · (θ − threshold)` for acting. Here `θ` is the hours change
(negative when families work less) and the threshold is `−150`, so enacting is worth
the hours the plan comes in *under* the memorandum's budget, at the Treasury's price per
hour across every eligible family.

In [ ]:
value = ValuePerOutcome(
    value=W.VALUE_PER_HOUR, outcome_unit="hour", numeraire="USD",
    source="Treasury memorandum, March 1968: forgone tax plus replacement transfer per hour of work",
)
decision = DecisionSpec(
    name="enact_national_plan", threshold=-W.HOURS_THRESHOLD,
    value_per_outcome_unit=value.value * W.N_ELIGIBLE, numeraire="USD",
)
print(value.ledger_line().statement)
print(f"\ndecision '{decision.name}': enact if the hours change exceeds {decision.threshold:.0f} {W.OUTCOME_UNIT}")
print(f"one hour per family across the eligible population is worth "
      f"{decision.value_per_outcome_unit / 1e6:,.1f} million USD a year")

## 3 · What Congress already has, and what it is worth

There is a survey. Two thousand low-income families, each facing some effective tax
rate on its earnings and some welfare guarantee, because of where it lives and what
it already receives. Nobody assigned any of it. The obvious regression — hours on the
tax rate — has been run many times, and its slope is the number the literature quotes.

In [ ]:
survey = W.prior_record(seed=1968)
print(survey[["tax_rate", "guarantee", "schooling", "hours"]].describe().loc[["mean", "std", "min", "max"]].round(2))

naive: LinearEstimate = ols(survey, y="hours", x="tax_rate")
adjusted = ols(survey, y="hours", x="tax_rate", covariates=["schooling", "guarantee"])
# The slope of the *true* response in the tax rate, at the survey's own mean plan.
truth_slope = -(W.TRUTH.substitution_effect + W.TRUTH.interaction * survey.guarantee.mean()
                + 2 * W.TRUTH.tax_curvature * survey.tax_rate.mean())
table(
    [
        ["naive", f"{naive.estimate:,.0f} ± {naive.se:,.0f}"],
        ["adjusted for schooling and the guarantee", f"{adjusted.estimate:,.0f} ± {adjusted.se:,.0f}"],
        ["the truth, at the survey's mean plan", f"{truth_slope:,.0f}"],
    ],
    headers=("estimator", "hours per unit of tax rate"),
)
print(f"\nthe naive slope is {naive.estimate / truth_slope:.1f}x the truth; adjusting for the one proxy "
      f"in the file leaves {adjusted.estimate / truth_slope:.1f}x")

In [ ]:
fig = figure(
    title="Two thousand families, nobody randomized: the slope is not the effect",
    subtitle="annual hours against the effective tax rate each family faces; colour is the confounder",
    x_title="effective tax rate on earnings", y_title="annual hours worked", height=440,
)
fig.add_scatter(
    x=survey.tax_rate, y=survey.hours, mode="markers", name="a family in the survey",
    marker={"size": 6, "color": survey.taste_unobserved, "colorscale": "RdYlBu", "reversescale": True,
            "opacity": 0.75, "line": {"width": 0.3, "color": "white"},
            "colorbar": {"title": "taste for<br>work<br>(unobserved)", "thickness": 12, "len": 0.7}},
)
grid = np.linspace(0.1, 0.8, 100)
fitted = float(survey.hours.mean()) + naive.estimate * (grid - float(survey.tax_rate.mean()))
fig.add_scatter(x=grid, y=fitted, name="the regression everyone runs", line={"color": W.DECISION_COLOR, "width": 3})
causal = (float(survey.hours.mean())
          + W.hours_change(float(survey.guarantee.mean()), grid)
          - float(W.hours_change(float(survey.guarantee.mean()), float(survey.tax_rate.mean()))))
fig.add_scatter(x=grid, y=causal, name="the causal curve, at the mean guarantee",
                line={"color": W.TRUTH_COLOR, "width": 3, "dash": "dash"})
caption(fig, "The colour is the confounder. Families who would work more anyway earn more, face "
             "lower effective rates, and sit up and to the left; the regression runs through that "
             "diagonal and reports a slope nearly four times the causal one. Adjusting for "
             "schooling — the only proxy in the file — moves it a third of the way back.")

## 4 · The graph says so, before the data does

The bias above is not a small-sample problem and no amount of care with the columns
in the survey fixes it. `axiom.identify` says why: write the graph down and ask.

In [ ]:
graph = CausalGraph.from_edges(
    "taste -> hours, taste -> tax_rate, taste -> guarantee, taste -> schooling, "
    "guarantee -> tax_rate, tax_rate -> hours, guarantee -> hours",
    unmeasured=["taste"], name="what_the_survey_knows",
)
verdict = identify(graph, "tax_rate", "hours")
print("status                  :", verdict.status)
print("route                   :", verdict.route)
print("admissible sets         :", adjustment_sets(graph, "tax_rate", "hours") or "none")
print("needs an unmeasured node:", requires_unmeasured(graph, "tax_rate", "hours"))
print("\nassumptions it would take to believe the survey number:")
table([[a.state, a.name, a.statement] for a in verdict.verdict.assumptions],
      headers=("state", "assumption", "statement"))
causal_graph(graph, height=340)

`status='downgraded'`, route `backdoor_unmeasured`, and **no admissible adjustment set
exists**. The path `tax_rate <- taste -> hours` stays open whatever is conditioned on,
because taste is not in the file; schooling is a proxy for it, and a proxy blocks a path
only if it *is* the confounder.

That verdict is the whole argument for spending money on an experiment. It is also
the reason the experiment has to assign plans to families: randomization deletes every
arrow into the plan and leaves a graph with nothing to adjust for.

In [ ]:
randomized = CausalGraph.from_edges(
    "taste -> hours, taste -> schooling, tax_rate -> hours, guarantee -> hours",
    unmeasured=["taste"], name="what_an_experiment_would_know",
)
for dose in ("guarantee", "tax_rate"):
    v = identify(randomized, dose, "hours")
    print(f"after randomization, {dose:9s}: {v.status} | route {v.route} | adjust for {v.adjustment_set or 'nothing'}")

## 5 · The estimand the decision needs

The Treasury's number is a **contrast**: hours under the plan against hours under no
plan, per family head, over the years the plan is promised, in the population a
national plan would reach. Each of those is a facet of the estimand and every one of
them will matter later: the *window* when Seattle-Denver enrols some families for five
years, the *reference* when New Jersey's control families gain a welfare alternative a
year in, the *population* when the experiments turn out to have enrolled city families
and the plan is national.

It is declared against the **assigned** plan. Families will not all receive the plan as
written — Gary's will under-report their earnings and be taxed on less — and what a
family receives is a consequence of what it was assigned, so conditioning on it would
undo the randomization.

In [ ]:
contrast = W.national_estimand()
print("estimand     :", contrast.name)
print("content hash :", contrast.content_hash()[:16])
print("intervention :", contrast.intervention.doses, f"({contrast.intervention.version})")
print("reference    :", contrast.reference.doses, f"({contrast.reference.version})")
print("population   :", contrast.population.name)
print("window       :", contrast.window, "| level:", contrast.level.unit)
print("dimension    :", contrast.dimension, "| unit:", contrast.outcome.unit)

five_year = W.national_estimand(years=5, name="hours_under_national_plan_5y")
national = W.national_estimand(population=W.eligible_population(national=True), name="hours_under_national_plan_us")
print("\nthe same contrast promised for five years differs in exactly one facet:", contrast.differing_facets(five_year))
print("the same contrast over the national population differs in exactly one facet:", contrast.differing_facets(national))

## 6 · What is believed now

Two things in the literature bear on the national plan. A 1962 cross-sectional study
of hours against welfare generosity, and a 1966 study exploiting differences in state
tax rates. Both report something like the contrast at a plan near the one on the
table, both are old, and `prior_from_history` discounts them by age and combines them.

In [ ]:
history = [
    StudySummary(name="cross_section_1962", treatment="plan", estimate=-230.0, se=110.0,
                 periods_ago=6.0, definition="wald"),
    StudySummary(name="state_rates_1966", treatment="plan", estimate=-95.0, se=90.0,
                 periods_ago=2.0, definition="wald"),
]
prior_mean, prior_sd = prior_from_history(history, half_life_periods=8.0)
print(f"prior on the hours change under the national plan: {prior_mean:.0f} ± {prior_sd:.0f} {W.OUTCOME_UNIT}")
print(f"the truth at a typical site, for the record: {float(W.hours_change(G_NAT, T_NAT)):.0f}")
print(f"the prior sits {(prior_mean - decision.threshold) / prior_sd:+.2f} prior sds from the threshold")

rng = np.random.default_rng(1)
fig = density(
    {"prior from two studies": rng.normal(prior_mean, prior_sd, 40_000)},
    ref=decision.threshold, ref_label="the Treasury's threshold",
    title="What is believed before the experiments",
    subtitle=f"the change in annual hours under G = {G_NAT}, t = {T_NAT}; the two studies it comes from are marked",
    x_title=f"hours change under the national plan ({W.OUTCOME_UNIT})",
)
for study in history:
    mark_x(fig, study.estimate, text=study.name, color=W.ACCENT, y=0.55)
caption(fig, "The prior straddles the threshold almost exactly. That is the worst place for a "
             "prior to sit when a decision has to be made on it, and the best place for it to "
             "sit when somebody is deciding whether to run an experiment.")

## 7 · What resolving it is worth

`evpi_gaussian` prices perfect information against a prior and a decision: the expected
cost of choosing under uncertainty rather than knowing. `evoi_gaussian` does the same for
an experiment that would only pin the number to some standard error.

In [ ]:
evpi = evpi_gaussian(decision, prior_mean, prior_sd)
print(f"EVPI on 'enact the plan on the table': {evpi / 1e6:,.0f} million USD a year")
rows = []
for se in (60.0, 40.0, 20.0, 10.0):
    result: EVOIResult = evoi_gaussian(decision, prior_mean, prior_sd, se)
    rows.append([f"±{se:.0f}", f"{result.evsi / 1e6:,.0f}", f"{result.evsi / evpi:.0%}",
                 f"{result.preposterior_sd:.0f}"])
table(rows, headers=("an experiment that pins the hours change to", "EVSI (million USD/yr)",
                     "of perfect", "preposterior sd"))

Over a quarter of a billion dollars a year, against experiments that will cost a few
tens of millions in total. And the value is nearly all captured by an experiment that
pins the number to twenty hours — which, as notebook 2 will show, four sites can buy.

There is a second question hiding in the first. Congress will not enact *exactly* the
plan on the table; the Family Assistance Plan that actually reached the floor in 1970
was `G = 0.42, t = 0.50`, and the counter-proposals ran up to `G = 1.0`. The decision is
therefore not one contrast but a region of them, and the question is whether the
experiments have to measure the region or can get away with a point. Price the same
go/no-go at three guarantees, with the prior scaled to each — an assumption the
experiments will have to remove, and the reason the estimand is a surface.

In [ ]:
rows, labels, values = [], [], []
for g in (0.5, G_NAT, 1.0):
    scale = g / G_NAT
    mean_g, sd_g = prior_mean * scale, prior_sd * scale
    threshold_g = -W.HOURS_THRESHOLD
    d_g = decision.model_copy(update={"threshold": threshold_g})
    e = evpi_gaussian(d_g, mean_g, sd_g)
    cost = W.plan_cost(g, T_NAT) * W.N_ELIGIBLE
    rows.append([W.plan_label(g, T_NAT), f"{mean_g:.0f} ± {sd_g:.0f}", f"{cost / 1e9:,.1f} bn",
                 f"{e / 1e6:,.0f}"])
    labels.append(W.plan_label(g, T_NAT)); values.append(e / 1e6)
table(rows, headers=("plan", "prior on the hours change", "programme cost/yr",
                     "EVPI (million USD/yr)"))
fig = compare(
    labels, values, highlight=W.plan_label(G_NAT, T_NAT), value_fmt="{:,.0f}",
    title="The decision is a region, not a point",
    subtitle="EVPI of the go/no-go at three guarantees Congress has discussed, million USD a year",
    x_title="EVPI (million USD/yr)",
)
caption(fig, "At the least generous plan the prior sits clear of the threshold and there is "
             "little to learn; at the two plans anyone is arguing about, the uncertainty is worth "
             "over a quarter of a billion a year each. An experiment that tested one plan would "
             "leave the other exactly where the literature left it — which is why notebook 2 "
             "allocates families across a lattice of plans rather than into two arms.")

## What notebook 1 established

1. **The dose is two numbers, and a plan is a straight line.** `G` and `t` are
   dimensionless by declaration, the transfer schedule is arithmetic, and the plan on
   the table costs nearly eight billion dollars a year before anybody changes how
   much they work.
2. **The survey cannot answer the question.** `identify` returns `downgraded` on the
   `backdoor_unmeasured` route with no admissible adjustment set, and the naive slope
   is nearly four times the causal one in the direction that makes the plan look
   unaffordable.
3. **The estimand is declared** — a contrast against the *assigned* plan, per family
   head, over three years, in the population the experiments will enrol — with a
   content hash, before any experiment exists. Its window, reference and population
   facets are the ones the next four notebooks will have to defend.
4. **The uncertainty is worth 270 million dollars a year** to resolve, 97 % of it
   captured by an experiment that pins the hours change to twenty hours — and it is
   worth as much again at the more generous plan Congress has discussed, and almost
   nothing at the less generous one. The experiments have to measure a surface, and
   the interesting part of it is at and above the plan on the table.

In [ ]:
carry = {
    "prior_mean": prior_mean, "prior_sd": prior_sd, "evpi_usd_per_year": float(evpi),
    "decision": decision.to_json(), "estimand": contrast.to_json(),
}
table([[key, f"{item:,.2f}" if isinstance(item, float) else "<spec>"] for key, item in carry.items()],
      headers=("carried into notebook 2", "value"))